# Lecture 08 - Solutions to Part B (Python Applications)

This notebook gives worked Python solutions for Exercises 13-18. The focus is hypothesis testing for a mean return using the normal approximation introduced in the lecture.

The data file is expected in the same folder as this notebook. If you store it elsewhere, change only `data_path`.

## Setup

Run this cell first. It loads Apple prices, computes daily returns from adjusted close, and defines small helper functions for normal p-values.

In [ ]:
import numpy as np
import pandas as pd
from math import erf, sqrt

pd.set_option("display.float_format", "{:.6f}".format)

data_path = "./"  # Use your data here. Keep the final slash.

aapl_raw = pd.read_csv(data_path + "Lecture 08 aapl daily 2012 2021.csv")
aapl_raw["Date"] = pd.to_datetime(aapl_raw["Date"])
aapl_raw = aapl_raw.sort_values("Date")

aapl = aapl_raw.copy()
aapl["daily_return"] = aapl["Adj Close"].pct_change()
aapl = aapl.dropna(subset=["daily_return"]).reset_index(drop=True)

def normal_cdf(x):
    return 0.5 * (1 + erf(x / sqrt(2)))

def normal_sf(x):
    return 1 - normal_cdf(x)

def mean_test(series, benchmark=0.0, alternative="greater"):
    r = pd.to_numeric(series, errors="coerce").dropna()
    n = r.count()
    mean = r.mean()
    sd = r.std(ddof=1)
    se = sd / np.sqrt(n)
    z = (mean - benchmark) / se
    if alternative == "greater":
        p_value = normal_sf(z)
    elif alternative == "less":
        p_value = normal_cdf(z)
    elif alternative == "two-sided":
        p_value = 2 * min(normal_cdf(z), normal_sf(z))
    else:
        raise ValueError("alternative must be 'greater', 'less', or 'two-sided'")
    return pd.Series({
        "n": n,
        "mean": mean,
        "sd": sd,
        "se": se,
        "benchmark": benchmark,
        "z_stat": z,
        "p_value": p_value,
        "alternative": alternative,
    })

aapl.head()

---

## Exercise 13 - Load Prices and Compute Returns

In [ ]:
summary_13 = pd.Series({
    "start_date": aapl["Date"].min(),
    "end_date": aapl["Date"].max(),
    "n_returns": aapl["daily_return"].count(),
    "mean_daily_return": aapl["daily_return"].mean(),
    "sd_daily_return": aapl["daily_return"].std(ddof=1),
    "se_daily_return": aapl["daily_return"].std(ddof=1) / np.sqrt(aapl["daily_return"].count()),
})

summary_13

---

## Exercise 14 - One-Sided Test for Positive Expected Return

We test:

$$
H_0:\mu\leq0,\qquad H_1:\mu>0.
$$

In [ ]:
test_positive = mean_test(aapl["daily_return"], benchmark=0.0, alternative="greater")
test_positive

In [ ]:
alpha = 0.05
decision_positive = "reject H0" if test_positive["p_value"] <= alpha else "fail to reject H0"

pd.Series({
    "alpha": alpha,
    "p_value": test_positive["p_value"],
    "decision": decision_positive,
})

**Interpretation.** The test evaluates whether the sample provides strong evidence that expected daily return is positive. The p-value should be compared with the chosen significance level before writing the conclusion.

---

## Exercise 15 - Two-Sided Test Against Zero

We test:

$$
H_0:\mu=0,\qquad H_1:\mu\neq0.
$$

In [ ]:
test_two_sided = mean_test(aapl["daily_return"], benchmark=0.0, alternative="two-sided")
test_two_sided

In [ ]:
decision_two_sided = "reject H0" if test_two_sided["p_value"] <= alpha else "fail to reject H0"

pd.Series({
    "alpha": alpha,
    "p_value": test_two_sided["p_value"],
    "decision": decision_two_sided,
})

**Interpretation.** The two-sided test treats unusually negative and unusually positive sample means as evidence against the benchmark. Its p-value is therefore different from the one-sided p-value.

---

## Exercise 16 - Testing Against an Annual Return Benchmark

The annual benchmark is 10%. Using the simple lecture approximation, the daily benchmark is \(0.10/252\).

In [ ]:
daily_benchmark = 0.10 / 252

test_annual_benchmark = mean_test(
    aapl["daily_return"],
    benchmark=daily_benchmark,
    alternative="greater",
)

test_annual_benchmark

In [ ]:
decision_annual_benchmark = "reject H0" if test_annual_benchmark["p_value"] <= alpha else "fail to reject H0"

pd.Series({
    "daily_benchmark": daily_benchmark,
    "alpha": alpha,
    "p_value": test_annual_benchmark["p_value"],
    "decision": decision_annual_benchmark,
})

**Interpretation.** This test asks whether the sample provides evidence that expected daily return is above the daily equivalent of a 10% annual benchmark. This is a simple linear benchmark, not a full investment recommendation.

---

## Exercise 17 - Sensitivity to the Significance Level

In [ ]:
alpha_rows = []
for level in [0.10, 0.05, 0.01]:
    alpha_rows.append({
        "alpha": level,
        "p_value": test_positive["p_value"],
        "decision": "reject H0" if test_positive["p_value"] <= level else "fail to reject H0",
    })

alpha_table = pd.DataFrame(alpha_rows)
alpha_table

Changing alpha changes the rejection rule. A smaller alpha requires stronger evidence before rejecting the null hypothesis.

---

## Exercise 18 - Short Statistical Memo

In [ ]:
memo = f"""
Using Apple daily adjusted closing prices from {summary_13['start_date'].date()} to {summary_13['end_date'].date()},
we computed daily simple returns and tested whether the expected daily return was positive.
The null hypothesis is H0: mu <= 0 and the alternative is H1: mu > 0.
The sample mean daily return is {test_positive['mean']:.6f}, with standard error {test_positive['se']:.6f}.
The resulting z-statistic is {test_positive['z_stat']:.3f} and the one-sided p-value is {test_positive['p_value']:.4f}.
At the 5% significance level, the decision is to {decision_positive}.
This is a statistical statement about the historical sample period. It does not prove future positive returns,
and it ignores investment frictions such as transaction costs, risk, taxes, and implementation constraints.
""".strip()

print(memo)